<a href="https://colab.research.google.com/github/anjalidrj/GO-analysis-of-peptides/blob/main/fetch_go_terms_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fetch GO terms for transcript hits
Upload your Excel file, run cells top to bottom. If Colab disconnects partway through, just re-run — it resumes from the checkpoint file automatically.

In [ ]:
# Cell 1: install dependencies
!pip install -q biopython requests pandas openpyxl

In [ ]:
# Cell 2: upload your Excel file
from google.colab import files
uploaded = files.upload()  # select your .xlsx file when prompted
INPUT_FILE = list(uploaded.keys())[0]
print('Uploaded:', INPUT_FILE)

In [ ]:
# Cell 3: config
import os, time, requests
import pandas as pd
from Bio import Entrez

SHEET_NAME = 1
TRANSCRIPT_COL = 'transcript_id'
OUTPUT_FILE = 'hits_with_GO_terms.xlsx'
CHECKPOINT_FILE = 'go_lookup_checkpoint.csv'

Entrez.email = 'anjali.d@stringbio.com'  # <-- put your real email here (required by NCBI)
Entrez.api_key = '4b6a3c9a2224f25ec40bc9ecec1efb283809'

NCBI_BATCH_SIZE = 20
REQUEST_DELAY = 0.15
CHECKPOINT_EVERY = 5

In [ ]:
# Cell 4: helper functions

import re
import requests

EUTILS_BASE = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
HEADERS = {
    'User-Agent': 'tomato-peptide-go-lookup/1.0',
    'Connection': 'close',
    'Accept-Encoding': 'identity',  # avoid gzip/chunked decoding issues that can truncate responses
}


def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


def _post_with_retries(url, data, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(url, data=data, headers=HEADERS, timeout=60)
            r.raise_for_status()
            return r
        except Exception as e:
            wait = 2 ** attempt
            print(f'    [request failed, attempt {attempt}/{max_retries}]: {e} -- retrying in {wait}s')
            time.sleep(wait)
    return None


def batch_transcript_to_protein(transcript_ids):
    """For a batch of transcript IDs, return {transcript_id: protein_accession or None}.
    Fetches GenBank flatfile records directly (single efetch call, no elink step)
    and parses the /protein_id="..." field out of each record's CDS feature."""
    result = {tid: None for tid in transcript_ids}

    efetch_data = {
        'db': 'nuccore',
        'id': transcript_ids,
        'rettype': 'gb',
        'retmode': 'text',
        'api_key': Entrez.api_key,
        'email': Entrez.email,
    }
    r = _post_with_retries(f'{EUTILS_BASE}/efetch.fcgi', efetch_data)
    if r is None:
        print('  [efetch batch failed after retries, skipping this batch]')
        return result

    text = r.text
    records = text.split('\n//\n')

    for rec in records:
        if not rec.strip():
            continue
        ver_match = re.search(r'^VERSION\s+(\S+)', rec, re.MULTILINE)
        if not ver_match:
            continue
        tid = ver_match.group(1)
        if tid not in result:
            continue
        prot_match = re.search(r'/protein_id="([^"]+)"', rec)
        result[tid] = prot_match.group(1) if prot_match else None

    return result


def map_to_uniprot(protein_accessions):
    if not protein_accessions:
        return {}

    strip_map = {}
    for acc in protein_accessions:
        stripped = re.sub(r'\.\d+$', '', acc)
        strip_map.setdefault(stripped, []).append(acc)
    stripped_list = list(strip_map.keys())

    mapping = {}
    for batch_num, batch in enumerate(chunked(stripped_list, 5000), 1):
        print(f'  Submitting batch {batch_num} ({len(batch)} accessions)...')
        run_url = 'https://rest.uniprot.org/idmapping/run'
        resp = requests.post(
            run_url,
            data={'from': 'RefSeq_Protein', 'to': 'UniProtKB', 'ids': ','.join(batch)},
            headers=HEADERS,
        )
        resp.raise_for_status()
        job_id = resp.json()['jobId']

        status_url = f'https://rest.uniprot.org/idmapping/status/{job_id}'
        max_wait_seconds = 300  # 5 min timeout per batch
        waited = 0
        while waited < max_wait_seconds:
            r = requests.get(status_url, headers=HEADERS)
            data = r.json()
            status = data.get('jobStatus', 'UNKNOWN')
            if status == 'FINISHED' or 'results' in data:
                print(f'    job {job_id} finished after {waited}s')
                break
            print(f'    job {job_id} status: {status} (waited {waited}s)')
            time.sleep(5)
            waited += 5
        else:
            print(f'    [WARNING] job {job_id} timed out after {max_wait_seconds}s, skipping this batch')
            continue

        results_url = f'https://rest.uniprot.org/idmapping/results/{job_id}?size=500'
        page_num = 0
        while results_url:
            r = requests.get(results_url, headers=HEADERS)
            r.raise_for_status()
            page_data = r.json()
            page_results = page_data.get('results', [])
            page_num += 1
            print(f'    results page {page_num}: {len(page_results)} matches')
            for item in page_results:
                stripped_id = item['from']
                uniprot_id = item['to']
                for orig in strip_map.get(stripped_id, []):
                    mapping[orig] = uniprot_id
            results_url = r.links.get('next', {}).get('url')

    return mapping


def get_go_terms(uniprot_acc):
    url = f'https://rest.uniprot.org/uniprotkb/{uniprot_acc}.json'
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()
        go_f, go_p, go_c = [], [], []
        for xref in data.get('uniProtKBCrossReferences', []):
            if xref.get('database') == 'GO':
                for prop in xref.get('properties', []):
                    if prop.get('key') == 'GoTerm':
                        term = prop.get('value', '')
                        if term.startswith('F:'):
                            go_f.append(term[2:])
                        elif term.startswith('P:'):
                            go_p.append(term[2:])
                        elif term.startswith('C:'):
                            go_c.append(term[2:])
        return {
            'GO_Molecular_Function': '; '.join(go_f),
            'GO_Biological_Process': '; '.join(go_p),
            'GO_Cellular_Component': '; '.join(go_c),
        }
    except Exception as e:
        print(f'  [UniProt lookup failed for {uniprot_acc}]: {e}')
        return {'GO_Molecular_Function': '', 'GO_Biological_Process': '', 'GO_Cellular_Component': ''}


In [ ]:
# Cell 5: load data + resume support
df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
all_transcripts = df[TRANSCRIPT_COL].dropna().unique().tolist()
print(f'Found {len(all_transcripts)} unique transcript IDs.')

if os.path.exists(CHECKPOINT_FILE):
    checkpoint_df = pd.read_csv(CHECKPOINT_FILE)
    done_transcripts = set(checkpoint_df[TRANSCRIPT_COL])
    transcript_to_protein = dict(zip(checkpoint_df[TRANSCRIPT_COL], checkpoint_df['protein_accession']))
    print(f'Resuming: {len(done_transcripts)} transcripts already resolved in checkpoint.')
else:
    checkpoint_df = pd.DataFrame(columns=[TRANSCRIPT_COL, 'protein_accession'])
    done_transcripts = set()
    transcript_to_protein = {}

remaining = [t for t in all_transcripts if t not in done_transcripts]
print(f'{len(remaining)} transcripts left to resolve via NCBI.')

In [ ]:
# Cell 6: Step 1 -- batched transcript -> protein accession, with checkpointing
batches = list(chunked(remaining, NCBI_BATCH_SIZE))
new_rows = []
for i, batch in enumerate(batches, 1):
    print(f'[NCBI batch {i}/{len(batches)}] resolving {len(batch)} transcripts...')
    batch_result = batch_transcript_to_protein(batch)
    transcript_to_protein.update(batch_result)
    for tid, acc in batch_result.items():
        new_rows.append({TRANSCRIPT_COL: tid, 'protein_accession': acc})
    time.sleep(REQUEST_DELAY)

    if i % CHECKPOINT_EVERY == 0 or i == len(batches):
        checkpoint_df = pd.concat([checkpoint_df, pd.DataFrame(new_rows)], ignore_index=True)
        checkpoint_df.to_csv(CHECKPOINT_FILE, index=False)
        new_rows = []
        print(f'  checkpoint saved ({len(checkpoint_df)} total resolved)')

In [ ]:
import requests
try:
    r = requests.get('https://eutils.ncbi.nlm.nih.gov/entrez/eutils/einfo.fcgi', timeout=15)
    print('Status:', r.status_code)
    print('Length:', len(r.content))
except Exception as e:
    print('Failed:', e)

In [ ]:
# Cell 7: Step 2 -- protein -> UniProt
protein_accs = [p for p in transcript_to_protein.values() if isinstance(p, str) and p]
print(f'Mapping {len(protein_accs)} protein accessions to UniProt...')
protein_to_uniprot = map_to_uniprot(protein_accs)

In [ ]:
sample_accs = [p for p in protein_accs[:10]]  # or however your list variable is named
print(sample_accs)
print(f"Total submitted: {len(protein_accs)}")
print(f"Total matched: {len(protein_to_uniprot)}")
print(f"Match rate: {len(protein_to_uniprot)/len(protein_accs)*100:.1f}%")

In [ ]:
# Cell 8: Step 3 -- UniProt -> GO terms
unique_uniprot = list(set(protein_to_uniprot.values()))
print(f'Fetching GO terms for {len(unique_uniprot)} UniProt entries...')
uniprot_go = {}
for i, uacc in enumerate(unique_uniprot, 1):
    if i % 100 == 0:
        print(f'  [{i}/{len(unique_uniprot)}] GO terms fetched so far...')
    uniprot_go[uacc] = get_go_terms(uacc)
    time.sleep(0.1)

In [ ]:
# Cell 9: Step 4 -- merge everything back into your original data and save
records = []
for tid, protein_acc in transcript_to_protein.items():
    uniprot_acc = protein_to_uniprot.get(protein_acc) if isinstance(protein_acc, str) else None
    go = uniprot_go.get(uniprot_acc, {'GO_Molecular_Function': '', 'GO_Biological_Process': '', 'GO_Cellular_Component': ''})
    records.append({TRANSCRIPT_COL: tid, 'protein_accession': protein_acc, 'uniprot_accession': uniprot_acc, **go})

go_table = pd.DataFrame(records)
merged = df.merge(go_table, on=TRANSCRIPT_COL, how='left')
merged.to_excel(OUTPUT_FILE, index=False)
print(f'Done. Output written to {OUTPUT_FILE}')

In [ ]:
# Cell 10: download the result
from google.colab import files
files.download(OUTPUT_FILE)

In [ ]:
print("Total rows:", len(merged))
print("Rows with protein_accession filled:", merged['protein_accession'].notna().sum())
print("Rows with uniprot_accession filled:", merged['uniprot_accession'].notna().sum())
print("Rows with any GO term filled:", (merged['GO_Molecular_Function'] != '').sum())

print("\nSample of protein_accession values:")
print(merged['protein_accession'].dropna().unique()[:10])

print("\nSample of uniprot_accession values:")
print(merged['uniprot_accession'].dropna().unique()[:10])